# Harness Engineering: OpenCode Case Study
## Chapter 2: Instructions and Guardrails

In Chapter 1, you followed OpenCode's attempt to repair a grade calculation and checked its work. Here we start again from the original buggy program, using fresh copies that leave your Chapter 1 work untouched.

We already used permissions in Chapter 1. Now we will compare two runs of the same repair request: one allows changes to `grades.py`; the other blocks them. Both allow the agent to write a report in `review.md`.

**Will OpenCode block a change to `grades.py` even when we ask it to fix the bug? What will the agent report?**

**Part 1** runs the experiment. **Part 2** explains what instructions and permissions each contribute. **Part 3** follows them in OpenCode's source.

### Part 1: Give the Same Task Different Permissions
#### 1. What have we asked the agent to do?

This lab defaults to Ollama. Before running setup, choose a GPU such as T4 under **Runtime > Change runtime type**. Then run the setup in a fresh Colab runtime. It installs OpenCode `1.18.34` and copies the example into a new folder, without calling a model.

Compared with Chapter 1's project, this copy adds two files:

- `AGENTS.md` holds the testing and reporting requirements that we previously wrote in the repair request.
- `review.md` is where the agent will write its report, so we can compare its claims with the changes and tests.

Read these files alongside the program, data, and test below.

In [ ]:
#@title Prepare OpenCode and the example project
from pathlib import Path
import copy
import subprocess
import sys
import tempfile

REPO_URL = "https://github.com/SleepyLGod/AIST4020-Lab-Tutorials.git"
LAB_RELATIVE = Path("labs/opencode/guardrails")
LAB_DIR = next((p for p in (Path.cwd(), Path.cwd() / LAB_RELATIVE)
                if (p / "guardrails_support.py").is_file()), None)
if LAB_DIR is None:
    course = Path(tempfile.mkdtemp(prefix="aist4020-course-"))
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(course)], check=True)
    LAB_DIR = course / LAB_RELATIVE
sys.path.insert(0, str(LAB_DIR))
import guardrails_support as support
if Path(support.__file__).resolve() != (LAB_DIR / "guardrails_support.py").resolve():
    raise RuntimeError("Restart the runtime to load this lab's helper.")
lab = support.open_lab(LAB_DIR, version="1.18.34")


In [ ]:
support.show_project(lab.project)


Check the starting program yourself. The missing score should not count toward the average. The test should fail with `60.0 != 90.0`; the assertions confirm that we reproduced the bug.

In [ ]:
TEST_COMMAND = ["python3", "-m", "unittest", "-v"]
AVERAGE_COMMAND = [
    "python3", "-c",
    "from grades import average; value = average('scores.csv'); "
    "print('Average:', value); assert value == 90.0, value",
]
initial_test = lab.check(TEST_COMMAND)
assert not initial_test.timed_out and initial_test.returncode != 0
assert "60.0 != 90.0" in initial_test.stdout + initial_test.stderr


Keep Ollama or choose an API provider. Model calls are enabled below (`ENABLE_MODEL_CALLS = True`). API keys come from Colab Secrets or hidden input; `key_env` names the environment variable that holds your key. Do not replace it with the key itself. Hosted APIs may charge for several model calls per task.

Use only this disposable course project. We will test tool permissions, not a security sandbox.

With Ollama selected, the next cell installs and starts Ollama and downloads the model; allow time for the first run.

In [ ]:
BACKENDS = {
    "deepseek": dict(base_url="https://api.deepseek.com/v1",
                     model="deepseek-flash", key_env="DEEPSEEK_API_KEY"),
    "nvidia": dict(base_url="https://integrate.api.nvidia.com/v1",
                   model="z-ai/glm-5.3-flash", key_env="NVIDIA_NIM_API_KEY"),
    "ollama": dict(base_url="http://127.0.0.1:11434/v1",
                   model="qwen3.5:9b-q4_K_M", key_env=""),
    "custom": dict(base_url="", model="", key_env=""),
}

backend = "ollama"
ENABLE_MODEL_CALLS = True
lab.configure(backend, BACKENDS, enabled=ENABLE_MODEL_CALLS)


#### 2. What happens when the edit is allowed?

OpenCode needs to inspect the project, repair the program, and report the result. Allow it to:

- Read files (`read`), find files by name (`glob`), and search their contents (`grep`).
- Change `grades.py` and `review.md`, leaving the data, test, and instructions untouched.
- Run `python3 -m unittest -v`.

The `edit` permission below applies to both changing an existing file and writing a file with the `write` tool.

In [ ]:
EDIT_ALLOWED_PERMISSIONS = {
    "*": "deny",  # Deny actions unless a rule below allows them.
    "read": "allow",
    "glob": "allow",
    "grep": "allow",
    "edit": {
        "*": "deny",  # Deny edits to other files.
        # Later matching rules win: these allow edits to the two named files.
        # Each pair matches the filename or a path ending with it.
        "grades.py": "allow", "**/grades.py": "allow",
        "review.md": "allow", "**/review.md": "allow",
    },
    "bash": {"*": "deny", "python3 -m unittest -v": "allow"},
    "external_directory": "deny",
}
repair_request = "Fix the average calculation. Record what you did and the test results in review.md."

EDIT_ALLOWED_SETTINGS = {
    "permission": EDIT_ALLOWED_PERMISSIONS,
    "agent": {"build": {"steps": 12}},
}
edit_allowed_case = lab.new_case("Edit allowed", EDIT_ALLOWED_SETTINGS)


`EDIT_ALLOWED_SETTINGS` uses OpenCode's configuration fields: `permission` contains the rules, and `agent.build.steps` keeps the twelve-step setting from Chapter 1. `lab.new_case(...)` copies the project and saves these settings alongside the model connection. `OPENCODE_CONFIG` points to the printed configuration path.

`AGENTS.md` belongs in the project folder; permissions belong in `opencode.json`. The optional Skill later goes in `.opencode/skills/repair-check/SKILL.md`.

Run the request below. OpenCode loads `AGENTS.md` from the project, so we do not repeat its testing and reporting instructions in the request. The command below runs in `edit_allowed_case.project` and starts a new conversation. The helper prints its actions.

**If a run stops:** read the error first. The helper stops after five minutes and does not retry. If your settings are unchanged, rerun the cell that creates the case, then rerun its task. To change your API key or model, restart the Python session, then run from setup with the corrected settings and repeat both experiments.

In [ ]:
print(repair_request)
RUN_COMMAND = [str(lab.binary), "run", "--pure", "--format", "json",
               "--agent", "build", "--model", edit_allowed_case.model]
allowed_run_result = edit_allowed_case.run([*RUN_COMMAND, repair_request])


Run the test and check the average yourself. `edit_allowed_case.check(...)` executes your command directly from the notebook, outside OpenCode's tool permissions. These are your checks, not tests performed by the agent.

The display below shows the actions, changed lines, and `review.md`. The report may be wrong, so check its claims against the actual file changes and test results. If the report says the agent retested, can you find a test command after its last code change?

In [ ]:
allowed_test_result = edit_allowed_case.check(TEST_COMMAND)
allowed_average_result = edit_allowed_case.check(AVERAGE_COMMAND)
support.show_case(edit_allowed_case, allowed_test_result, allowed_average_result)


#### 3. What changes when we deny that edit?

Keep the model, request, and starting files the same. Change only the permission to edit `grades.py`; writing `review.md` is still allowed.

This run starts from another copy of the original buggy program and a new conversation, not the files changed in the previous run. Predict what its report will say if OpenCode blocks the repair.

In [ ]:
EDIT_DENIED_SETTINGS = copy.deepcopy(EDIT_ALLOWED_SETTINGS)
EDIT_DENIED_SETTINGS["permission"]["edit"]["grades.py"] = "deny"
EDIT_DENIED_SETTINGS["permission"]["edit"]["**/grades.py"] = "deny"

edit_denied_case = lab.new_case("Edit denied", EDIT_DENIED_SETTINGS)
assert edit_allowed_case.before == edit_denied_case.before  # The project files start identical.
denied_run_result = edit_denied_case.run([*RUN_COMMAND, repair_request])


Find a request to edit `grades.py` and read the response. If the edit was refused, look for a later action that writes to `review.md`. Then check the program and report below.

**No edit request?** An unchanged file alone does not show that permission blocked an edit. A refusal of another action, such as `ls -la`, does not show this either.

You can repeat the experiment with a fresh case, or follow the permission check in Part 3.2. Without an edit request, this run does not show whether that edit would be blocked.

In [ ]:
denied_test_result = edit_denied_case.check(TEST_COMMAND)
denied_average_result = edit_denied_case.check(AVERAGE_COMMAND)
support.show_case(edit_denied_case, denied_test_result, denied_average_result)


#### 4. What does the comparison show?

Compare the edit requests, file changes, and test results from your two runs. Did allowing the edit lead to a repair? Did denying it stop an attempted change?

If neither run fixes the bug, look for an edit request refused by the permission rule in the second run. That shows the rule blocked that action; allowing an edit does not guarantee a successful repair.

If the edit was blocked, the program may still return 60 and fail the test. That is a broken program, not an OpenCode crash. Read `review.md` too: could the agent explain the blocker even though it could not fix the code?

In [ ]:
support.compare_cases(
    edit_allowed_case, edit_denied_case,
    allowed_test=allowed_test_result, denied_test=denied_test_result,
    allowed_average=allowed_average_result, denied_average=denied_average_result,
)


#### 5. Can we reuse the repair steps? (Optional)

Read the files, run the test, repair the code, test again, and write a report: these steps can guide other repairs too. A **Skill** keeps instructions like these in a file the agent can load when needed. Our example is called `repair-check`.

Read it below. The steps overlap with `AGENTS.md`; this experiment shows how a Skill is loaded, not whether it improves the repair. In the optional run, `with_skill=True` copies this file to `.opencode/skills/repair-check/SKILL.md` in the new project. We allow that Skill and ask for it by name.

In [ ]:
support.show_details(
    "repair-check/SKILL.md",
    (LAB_DIR / "skills" / "repair-check" / "SKILL.md").read_text(),
    opened=True,
)


In [ ]:
RUN_SKILL_EXAMPLE = False

if RUN_SKILL_EXAMPLE:
    SKILL_SETTINGS = copy.deepcopy(EDIT_ALLOWED_SETTINGS)
    SKILL_SETTINGS["permission"]["skill"] = {"*": "deny", "repair-check": "allow"}
    skill_case = lab.new_case("With repair Skill", SKILL_SETTINGS, with_skill=True)
    print("Skill file:", skill_case.project / ".opencode/skills/repair-check/SKILL.md")
    skill_run_result = skill_case.run([*RUN_COMMAND, "Load the repair-check skill, then complete this task: " + repair_request])
    support.show_skill(skill_case)
    skill_test_result = skill_case.check(TEST_COMMAND)
    skill_average_result = skill_case.check(AVERAGE_COMMAND)
    support.show_case(skill_case, skill_test_result, skill_average_result)


If you ran the example, find the `skill` action and read the instructions it returned. Which actions followed? Loading the text does not run the tests or change the program; those steps need separate tool calls.

### Part 2: Guidance and Guardrails

**Instructions tell the model how to work.** In `AGENTS.md`, we asked for tests before and after a change. The model sees that request, but it may still skip a test. To find out, we read the actions it took.

**A Skill supplies instructions for a job.** `repair-check` groups the repair steps in one place. Loading it gives the model those steps; the tool permissions still decide whether it may carry them out.

**Permissions check a proposed action.** A model could work out the correct fix for `grades.py` and request an edit. With our deny rule, OpenCode returns an error before writing that change. The request to repair the program does not override the rule.

![An edit request is checked before OpenCode writes the file. An allowed edit returns a result; a denied edit returns an error.](https://raw.githubusercontent.com/SleepyLGod/AIST4020-Lab-Tutorials/main/labs/opencode/guardrails/assets/permission-check.png)

OpenCode supports three decisions: `allow` lets the action proceed, `deny` stops it, and `ask` requires approval. Our two runs use `allow` and `deny`.

**Independent checks tell us what happened.** The edit response shows whether a request was refused. The file comparison shows what changed. Our test checks the calculation. A blocked edit and a failed test can therefore both be expected: the restriction worked, but the repair remains unfinished.

These permissions are one kind of **guardrail**: a check or limit that controls what an agent receives, does, or returns. Other examples include hiding personal information before sending text to a model or requiring approval before sending an email. This lab focuses on tool actions. [More examples: LangChain Guardrails](https://docs.langchain.com/oss/python/langchain/guardrails/).

#### What would a sandbox add?

Suppose we allow `python3 -m unittest -v`. That command runs Python code. Checking the command does not, by itself, restrict every file or network address that the Python program can reach.

**Tool permissions** decide whether a requested action may start. A **sandbox** restricts what the running program can access, according to its configuration. They solve different parts of the problem.

OpenCode `1.18.34` does not provide security isolation through its permission system. We are observing action checks here, not testing a sandbox. See [OpenCode's security explanation](https://github.com/anomalyco/opencode/blob/v1.18.34/SECURITY.md); [Claude Code's sandbox documentation](https://code.claude.com/docs/en/sandboxing) gives an example of filesystem and network restrictions.

### Part 3: Find the Checks in the Source

Follow the same three questions: how do project instructions reach the model, where is an edit stopped, and what does loading a Skill return?

The excerpts use [OpenCode v1.18.34](https://github.com/anomalyco/opencode/tree/v1.18.34). `// ...` omits unrelated lines. Read `yield*` as "perform this operation and get its result"; focus on the order of the calls.

#### 1. How does AGENTS.md reach the model?

Our request says to fix the average and write a report. `AGENTS.md` includes these instructions:

```text
- Use python3 -m unittest -v before and after any implementation change.
- Report only actions you performed. List implementation and report changes separately.
- If an action is blocked, describe the blocker and what remains unfinished.
```

OpenCode looks for this filename and reads the instruction files it finds:

```ts
const instructionFiles = [
  "AGENTS.md",
  // ...
]
// ...
const system = Effect.fn("Instruction.system")(function* () {
  // ...
  const paths = yield* systemPaths()
  // ...
  const files = yield* Effect.forEach(Array.from(paths), read, { concurrency: 8 })
  // ...
  return [
    ...Array.from(paths).flatMap((item, i) => (files[i] ? [`Instructions from: ${item}\n${files[i]}`] : [])),
    // ...
  ]
})
```

*Source: [session/instruction.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/session/instruction.ts)*

Follow one file: `paths` includes the path to `AGENTS.md`; `read` puts its text in `files`. The returned text begins with `Instructions from:` and the filename, followed by our project instructions.

In the next snippet, follow our testing requirement through `instructions`: `instruction.system()` returns the text from `AGENTS.md` as part of `instructions`. The loop adds it to `system` and passes that guidance to `handle.process(...)`:

```ts
const [skills, env, instructions, mcpInstructions, modelMsgs] = yield* Effect.all([
  // ...
  instruction.system().pipe(Effect.orDie),
  // ...
])
const system = [
  // ...
  ...instructions,
  // ...
]
// ...
const result = yield* handle.process({
  // ...
  system,
  // ...
})
```

*Source: [session/prompt.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/session/prompt.ts)*

The two files use the name `system` differently: in `instruction.ts`, it is the function that reads the instructions; in `prompt.ts`, it is the array that collects instruction text for the model. Our testing requirement reaches the model as guidance; it does not force a test to run.

#### 2. Where does OpenCode stop an edit?

Suppose the model requests a change to `grades.py`. The edit tool checks permission before writing it:

```ts
yield* ctx.ask({
  permission: "edit",
  patterns: [path.relative(instance.worktree, filePath)],
  // ...
})
// ...
yield* afs.writeWithDirs(filePath, Bom.join(contentNew, desiredBom))
```

*Source: [tool/edit.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/tool/edit.ts)*

Here, `filePath` points to `grades.py`. `path.relative(...)` expresses it relative to the project, so the check can match our `"grades.py": "deny"` rule. `contentNew` holds the proposed new code; the last line writes it only if the permission check lets execution continue. Despite its name, `ctx.ask` can decide from the rules without asking a person.

Inside that check, `evaluate(...)` finds the last matching rule. Follow `rule.action`:

```ts
for (const pattern of request.patterns) {
  const rule = evaluate(request.permission, pattern, ruleset, approved)
  // ...
  if (rule.action === "deny") {
    return yield* new PermissionV1.DeniedError({
      // ...
    })
  }
  if (rule.action === "allow") continue
  // ...
}
```

*Source: [permission/index.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/permission/index.ts)*

`ruleset` contains the configured rules; `approved` holds any earlier approvals. For an edit of `grades.py` in our denied case, the matching rule says `deny`. The check returns `DeniedError`, so the write above is not reached. For `review.md`, our rule still says `allow`.

The refusal stops this edit, not necessarily the whole task. OpenCode [saves the tool error](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/session/processor.ts#L186) and [includes it in the next model input](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/session/message-v2.ts#L329). The model can then request an allowed action, such as writing `review.md` to explain the blocked repair. Check your action records to see whether it did.

This explains the specific edit refusal you looked for in Part 1. The [write tool](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/tool/write.ts) also checks the `edit` permission, so using `write` instead of `edit` does not bypass the file rule.

#### 3. What does loading a Skill return?

In the optional request, we named `repair-check`. That name becomes `params.name`. OpenCode finds the Skill and checks permission to load it:

```ts
const info = yield* skill
  .require(params.name)
  // ...

yield* ctx.ask({
  permission: "skill",
  patterns: [params.name],
  // ...
})
```

The returned `info` contains the Skill's name and content. The tool puts that content in its result:

```ts
return {
  title: `Loaded skill: ${info.name}`,
  output: [
    `<skill_content name="${info.name}">`,
    `# Skill: ${info.name}`,
    "",
    info.content.trim(),
    // ...
  ].join("\n"),
  // ...
}
```

*Source: [tool/skill.ts](https://github.com/anomalyco/opencode/blob/v1.18.34/packages/opencode/src/tool/skill.ts)*

`info.content.trim()` supplies the instructions you read in `SKILL.md`. They return as a tool result, just as test output did in Chapter 1. The model can use them to choose its next action. The code here returns text; it does not execute the repair steps.

The same request can lead to different outcomes because OpenCode checks the proposed actions. Instructions describe the work we want; permissions decide which actions may proceed. Our own tests and file checks tell us what was done.

To keep your records, run the optional cell below before disconnecting and deleting the Colab runtime. Review the downloaded files before sharing them.

In [ ]:
DOWNLOAD_RECORDS = False
if DOWNLOAD_RECORDS:
    lab.download_records()
